# Long Short-Term Memory (LSTM) Model Training for Normalized Difference Vegetation Index (NDVI) Anomaly Detection

This notebook trains LSTM neural networks to detect anomalies in NDVI time series.

## Objectives:
- Prepare sequenced data for LSTM training
- Build and train LSTM model
- Evaluate model performance
- Save trained model for production use

## 1. Initialize Environment and Import Libraries

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.insert(0, '../src')
from lstm_model import NDVIAnomalyLSTM
from utils import calculate_statistics, plot_anomaly_detection

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set up visualization
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ Libraries loaded successfully")
print(f"TensorFlow version: {tf.__version__}")
print(tf.keras.__version__)

## 2. Load Processed NDVI Data

In [ ]:
# Define paths
PROJECT_ROOT = Path('../').resolve()
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

# Load NDVI time series
ndvi_data = np.load(str(PROCESSED_DATA_DIR / 'ndvi_values.npy'))
ts_df = pd.read_csv(str(PROCESSED_DATA_DIR / 'ndvi_timeseries.csv'))

print(f"Loaded {len(ndvi_data)} NDVI observations")
print(f"Date range: {ts_df['start'].min()} to {ts_df['end'].max()}")
print(ts_df['median_ndvi'].describe())

In [ ]:
# Encode seasonality with cyclic features so month/phase wraps smoothly (e.g., Dec -> Jan).
t = np.arange(len(ndvi_data))

# Period=12 assumes a 12-step seasonal cycle.
# If your time step is biweekly, verify this period matches your intended seasonality.
sin_t = np.sin(2 * np.pi * t / 12)
cos_t = np.cos(2 * np.pi * t / 12)

# Final input features: [NDVI value, seasonal_sin, seasonal_cos]
features = np.stack([ndvi_data, sin_t, cos_t], axis=1)

print("Feature matrix shape (samples, features):", features.shape)

## 3. Prepare Data for LSTM

In [ ]:
# Initialize LSTM hyperparameters
sequence_length = 12   # Number of historical time steps per input sequence
forecast_horizon = 3   # Number of future steps to predict
lstm_units = 64        # Hidden units in LSTM layers

lstm_model = NDVIAnomalyLSTM(
    sequence_length=sequence_length,
    forecast_horizon=forecast_horizon,
    lstm_units=lstm_units
)

# Convert feature matrix into supervised sequences (X) and targets (y)
X, y = lstm_model.prepare_sequences(features)

print(f"Sequence length: {sequence_length}")
print(f"Forecast horizon: {forecast_horizon}")
print(f"\nX shape: {X.shape}  (samples, sequence_length, features)")
print(f"y shape: {y.shape}  (samples, forecast_horizon)")

# Chronological split (no shuffling) to avoid time leakage
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
#K.clear_session()

# ensure dtype
X_train = np.asarray(X_train, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)

y_train = np.asarray(y_train, dtype=np.float32)
y_test = np.asarray(y_test, dtype=np.float32)

# fix y shape
if y_train.ndim == 3:
    y_train = y_train.reshape(y_train.shape[0], y_train.shape[1])

if y_test.ndim == 3:
    y_test = y_test.reshape(y_test.shape[0], y_test.shape[1])

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

## 4. Build LSTM Model Architecture

In [ ]:
# Build model
lstm_model.build_model()

# Display model architecture
lstm_model.model.summary()

## 5. Train Model

In [ ]:
# Train the LSTM model
# Parameters:
# - X_train, y_train: Training data and labels
# - validation_split: Fraction of training data used for validation (e.g., 0.2 = 20%)
# - epochs: Number of times the model sees the entire training set
# - batch_size: Number of samples per gradient update
# - verbose: 1 = progress bar, 2 = one line per epoch, 0 = silent
# - callbacks: List of Keras callbacks (e.g., ModelCheckpoint)

# save the best model weights (lowest val_loss) during training
checkpoint_path = str(MODELS_DIR / "best_lstm_weights.weights.h5")

checkpoint = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
    verbose=1,
)

# Train the LSTM model
history = lstm_model.model.fit(
    X_train,
    y_train,
    validation_split=0.2,        # 20% of training data for validation
    epochs=50,                   # Train for 50 epochs
    batch_size=16,               # 16 samples per batch
    verbose=1,                   # Show progress bar
    callbacks=[checkpoint],      # Save best weights
)

lstm_model.history = history    # Save history for later analysis


# Load the best weights after training
lstm_model.model.load_weights(checkpoint_path)

print("✓ Model training complete (best weights loaded)")

## 6. Evaluation

In [ ]:
# Evaluate the trained LSTM model on the test set
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

# Keras evaluation returns the model loss and tracked metrics
test_loss, test_mae = lstm_model.model.evaluate(X_test, y_test, verbose=1)

print("Test loss (MSE):", test_loss)
print("Test MAE:", test_mae)

# Generate predictions for additional regression metrics
y_pred = lstm_model.model.predict(X_test, verbose=0)
print("Prediction shape:", y_pred.shape)

# Calculate standard regression metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

## 7. Plot Training History

In [ ]:
# Plot training history
history = lstm_model.history

if history is None:
    raise ValueError("No training history found. Train the model before plotting.")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(history.history.get('loss', []), label='Training Loss', linewidth=2)
axes[0].plot(history.history.get('val_loss', []), label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss (MSE)', fontsize=11)
axes[0].set_title('Model Loss During Training', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# MAE
if 'mae' in history.history:
    axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2)
if 'val_mae' in history.history:
    axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('MAE', fontsize=11)
axes[1].set_title('Model MAE During Training', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(MODELS_DIR / 'training_history.png'), dpi=200, bbox_inches='tight')
plt.show()

print("✓ Training history plot saved")

## 8. Visualization

In [ ]:
# Visualize LSTM Predictions vs Actual NDVI
plt.figure(figsize=(10, 5))
plt.plot(y_test[:, 0], label='Actual (t+1)', color='green')
plt.plot(y_pred[:, 0], '--', label='Predicted (t+1)', color='orange')
plt.title('NDVI Prediction (1-step ahead)')
plt.xlabel('Sample index')
plt.ylabel('NDVI')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Save Trained Model

In [ ]:
# Save model
model_path = str(MODELS_DIR / 'ndvi_anomaly_lstm.h5')
lstm_model.save_model(model_path)

# Save scaler for future predictions
import pickle
scaler_path = str(MODELS_DIR / 'ndvi_scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(lstm_model.scaler, f)

# Save model metadata
metadata = {
    'model_type': 'LSTM',
    'sequence_length': sequence_length,
    'forecast_horizon': forecast_horizon,
    'lstm_units': lstm_units,
    'epochs_trained': len(history.history['loss']),
    'test_mse': float(test_loss),
    'test_mae': float(test_mae),
    'test_rmse': float(rmse),
    'training_date': pd.Timestamp.now().isoformat()
}

import json
with open(str(MODELS_DIR / 'model_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"✓ Model saved to: {model_path}")
print(f"✓ Scaler saved to: {scaler_path}")
print(f"✓ Metadata saved to: {MODELS_DIR / 'model_metadata.json'}")

## Next Steps

1. Run **04_anomaly_detection.ipynb** to detect anomalies in new data
2. Implement real-time monitoring pipeline
3. Set up alerting system for fire risk detection